# 05 — CNN smoke test: does the data/model plumbing run end-to-end and can it overfit a tiny batch?

**Decision this feeds (`structuring-ml-projects` SKILL.md, cheap-proxy
ladder, `deep-learning-imaging.md`):** rung 0 (mandatory before any
longer run) — right shapes/dtypes end to end, and the model can overfit
a tiny labeled subset to near-zero loss. If rung 0 passes, rung 1 (a
10-20% stratified subset, one fold) checks the loss curve is sane. Note:
**this notebook does not decide anything about the model's real
performance** — only rung 3 (a full CV run, not built yet) clears the
gate against the classical baseline (`model.build_combat_baseline()`,
0.5290 log loss). See
`docs/superpowers/specs/2026-09-08-cnn-data-plumbing-design.md`.

**Data handling:** this notebook loads real `.nii.gz` volumes and
row-level labels, so per the AI-assistant data rule (`README.md`) it is
**[RUN ME]** — run it yourself, share back only the printed shapes and
loss values, not any per-row output.

## QC: does the fixed crop box actually contain the striatum?

`CROP_CENTER_MM`/`CROP_SIZE_MM` (`config.py`) were measured against a
different axis convention than `data.crop_or_pad` applies them in; this
cell is a cheap sanity check, not a proof. For a few real volumes it
prints the intensity-weighted center of mass of the *cropped* array as a
fraction of that array's shape per axis (0 = one edge, 1 = the other,
0.5 = geometric center) -- a striatum-centered crop should land close to
0.5 on each axis, not pinned near 0 or 1. **[RUN ME]** -- loads real
pixel data.

In [ ]:
# [RUN ME] -- loads real pixel data (no labels needed for this check).
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / "src"))

import numpy as np
import pandas as pd

import config
import data

qc_labels_df = pd.read_csv(config.TRAIN_LABELS_PATH)
qc_uids = qc_labels_df[config.UID_COLUMN].tolist()[:3]

for uid in qc_uids:
    volume = data.load_volume(uid)[0]  # drop channel dim -> (56, 30, 44)
    signal = np.clip(volume, 0, None)
    if signal.sum() <= 0:
        print(f"{uid}: no positive signal in cropped volume, cannot compute centroid")
        continue
    idx = np.indices(signal.shape, dtype=float)
    centroid_vox = [np.average(idx[axis], weights=signal) for axis in range(3)]
    centroid_frac = [c / (s - 1) for c, s in zip(centroid_vox, signal.shape)]
    print(f"{uid}: centroid as fraction of shape (0.5 = geometric center) = "
          f"{[round(f, 3) for f in centroid_frac]}")

In [ ]:
# [RUN ME] -- loads real pixel data + row-level labels.
import sys
import time
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / "src"))

import numpy as np
import pandas as pd
import torch

import config
import dataset
import model

torch.manual_seed(config.SEED)


class CachedDataset(torch.utils.data.Dataset):
    """In-memory cache wrapper, notebook-local only -- rung 0/1 revisit the
    same small subset every epoch, and DatParkinsonDataset re-reads +
    re-resamples from disk on every __getitem__ call with no caching of
    its own. Wrapping it here avoids ~30x redundant disk I/O for rung 0's
    24 volumes (and similarly for rung 1's larger subset). Not part of
    src/dataset.py: an unbounded in-memory cache is fine for a few hundred
    volumes but not a production choice for the full 1362 -- a real
    on-disk cache is the follow-up plan's job (see the final review's
    recommendations)."""

    def __init__(self, base_dataset):
        self.base = base_dataset
        self._cache = {}

    def __len__(self):
        return len(self.base)

    def __getitem__(self, idx):
        if idx not in self._cache:
            self._cache[idx] = self.base[idx]
        return self._cache[idx]


labels_df = pd.read_csv(config.TRAIN_LABELS_PATH)
rng = np.random.RandomState(config.SEED)
smoke_idx = rng.choice(len(labels_df), size=24, replace=False)
smoke_df = labels_df.iloc[smoke_idx]

smoke_ds = CachedDataset(dataset.DatParkinsonDataset(
    uids=smoke_df[config.UID_COLUMN].tolist(),
    labels=smoke_df[config.TARGET_COLUMN].tolist(),
))
loader = torch.utils.data.DataLoader(smoke_ds, batch_size=8, shuffle=True)

# .to(config.DEVICE): build_model() alone leaves the net on CPU -- without
# this, training silently runs on CPU even though config.DEVICE = "cuda".
net = model.build_model().to(config.DEVICE)
opt = torch.optim.Adam(net.parameters(), lr=config.LR)
loss_fn = torch.nn.BCEWithLogitsLoss()

print(f"rung 0: {len(smoke_ds)} labeled volumes, checking shapes + overfit")
print(f"training on device: {next(net.parameters()).device}")
for x, y in loader:
    print("batch shapes:", x.shape, x.dtype, y.shape, y.dtype)
    break

print("preprocessing (nibabel resample) dominates epoch 0's time; epochs "
      "after that reuse the in-memory cache and should be much faster.")
start = time.time()
losses = []
for epoch in range(30):
    epoch_loss = 0.0
    for x, y in loader:
        x, y = x.to(config.DEVICE), y.to(config.DEVICE)
        opt.zero_grad()
        loss = loss_fn(net(x), y)
        loss.backward()
        opt.step()
        epoch_loss += loss.item() * x.shape[0]
    losses.append(epoch_loss / len(smoke_ds))
    print(f"epoch {epoch}: loss = {losses[-1]:.4f} ({time.time() - start:.1f}s elapsed)")

print("rung 0 loss curve (first/mid/last):", losses[0], losses[len(losses) // 2], losses[-1])

In [ ]:
# [RUN ME] -- only run after rung 0 above shows the loss reaching near
# zero. Loads a larger real subset + labels.
import evaluate

# inplane_family isn't in train_labels.csv itself (it's derived from NIfTI
# headers) -- reuse notebooks/03's baseline_features.csv, which already
# has it per uid, rather than re-deriving it here.
family_df = pd.read_csv(config.DATA_PROCESSED / "baseline_features.csv")[
    [config.UID_COLUMN, "inplane_family"]
]
labeled_df = labels_df.merge(family_df, on=config.UID_COLUMN, how="inner")

rng2 = np.random.RandomState(config.SEED)
subset_idx = rng2.choice(len(labeled_df), size=int(0.15 * len(labeled_df)), replace=False)
subset_df = labeled_df.iloc[subset_idx].reset_index(drop=True)

folds = evaluate.make_folds(
    subset_df[config.TARGET_COLUMN].to_numpy(),
    subset_df["inplane_family"].to_numpy(),
    n_splits=config.N_FOLDS, random_state=config.SEED,
)
train_idx, val_idx = folds[0]

# num_workers=0 here, not config.NUM_WORKERS: with multiple worker
# processes each gets its own copy of CachedDataset, so the cache
# wouldn't be shared and would buy nothing -- the in-memory cache
# eliminates far more redundant I/O than 4-way parallelism would anyway,
# once past epoch 0.
train_ds = CachedDataset(dataset.DatParkinsonDataset(
    uids=subset_df.iloc[train_idx][config.UID_COLUMN].tolist(),
    labels=subset_df.iloc[train_idx][config.TARGET_COLUMN].tolist(),
))
val_ds = CachedDataset(dataset.DatParkinsonDataset(
    uids=subset_df.iloc[val_idx][config.UID_COLUMN].tolist(),
    labels=subset_df.iloc[val_idx][config.TARGET_COLUMN].tolist(),
))
train_loader = torch.utils.data.DataLoader(train_ds, batch_size=config.BATCH_SIZE, shuffle=True)
val_loader = torch.utils.data.DataLoader(val_ds, batch_size=config.BATCH_SIZE)

# .to(config.DEVICE): build_model() alone leaves the net on CPU -- without
# this, training silently runs on CPU even though config.DEVICE = "cuda".
net = model.build_model().to(config.DEVICE)
opt = torch.optim.Adam(net.parameters(), lr=config.LR)
loss_fn = torch.nn.BCEWithLogitsLoss()

# Rung 1 is a smoke-test proxy, not a real training run -- the spec calls
# for "few epochs", not config.EPOCHS (50, the full-training default), so
# this uses its own small local epoch count instead.
rung1_epochs = 15

print(f"rung 1: train={len(train_ds)}, val={len(val_ds)}, {rung1_epochs} epochs, 1 fold")
print(f"training on device: {next(net.parameters()).device}")
print("epoch 0 pays the full disk-read/resample cost for every volume; "
      "later epochs reuse the in-memory cache and should be much faster.")
start = time.time()
for epoch in range(rung1_epochs):
    net.train()
    for x, y in train_loader:
        x, y = x.to(config.DEVICE), y.to(config.DEVICE)
        opt.zero_grad()
        loss = loss_fn(net(x), y)
        loss.backward()
        opt.step()

    net.eval()
    val_probs, val_labels = [], []
    with torch.no_grad():
        for x, y in val_loader:
            x = x.to(config.DEVICE)
            val_probs.append(torch.sigmoid(net(x)).cpu().numpy())
            val_labels.append(y.numpy())
    val_probs = np.concatenate(val_probs)
    val_labels = np.concatenate(val_labels)
    val_loss = evaluate.log_loss_score(val_labels, val_probs)
    print(f"epoch {epoch}: val log loss = {val_loss:.4f} "
          f"({time.time() - start:.1f}s elapsed)")

print(f"rung 1 final val log loss: {val_loss:.4f} "
      f"(reference only -- classical build_combat_baseline() = 0.5290; "
      f"this is NOT a gate decision)")

**What we're looking for:** does the plumbing run end to end with the
right shapes, and can the model overfit a tiny labeled batch (rung 0)?
If so, is the rung-1 loss curve sane (not diverging, not stuck at the
base-rate loss)?

**What we found:**
- Rung 0 (24 volumes, batch size 8): batch shapes `[8, 1, 56, 30, 44]`
  float32 / labels `[8]` float32 — matches the expected 5D CNN input
  (batch, channel, D, H, W). Training loss went `0.937 → 0.291 (mid) →
  0.059 (last)` over 30 epochs, a clean monotonic-ish overfit to
  near-zero. Ran on `cuda:0`.
- Rung 1 (train=163, val=41, 15 epochs, 1 fold): val log loss is noisy
  across epochs (range ~0.39–0.83) but stays mostly well below the
  coin-flip baseline (ln 2 ≈ 0.693) from epoch 0 onward and never
  diverges. Final epoch: 0.5638 vs. classical `build_combat_baseline()`
  = 0.5290 (reference only, not a gate — see intro cell). The
  epoch-to-epoch noise is expected at this scale: 41 validation examples,
  no LR schedule/early stopping yet, and a single fold.

**Decision / next step:** Rung 0 passes cleanly (near-zero overfit loss
on the tiny subset, correct shapes/dtypes, runs on GPU) — no plumbing
bug to fix. Rung 1's curve is sane (non-diverging, consistently below
base-rate) — the proxy check that gates moving forward, not the real
performance gate. **Next: the follow-up spec** (full `train.py`,
augmentation, rung 2/3) per
`docs/superpowers/specs/2026-09-08-cnn-data-plumbing-design.md`.</cell id="c05find01">